# 03 — Analyse & KPIs
**Objectif** : Calculer tous les indicateurs clés et produire les visualisations finales pour le rapport et Power BI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

CLEANED_PATH = '../data/cleaned/'
DOCS_PATH    = '../docs/'
os.makedirs(DOCS_PATH, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
BLUE   = '#185FA5'
GREEN  = '#3B6D11'
RED    = '#A32D2D'
ORANGE = '#BA7517'
GRAY   = '#888780'

print('Librairies prêtes.')

In [ ]:
orders      = pd.read_csv(CLEANED_PATH + 'orders_clean.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items       = pd.read_csv(CLEANED_PATH + 'items_clean.csv')
customers   = pd.read_csv(CLEANED_PATH + 'customers_clean.csv')
reviews     = pd.read_csv(CLEANED_PATH + 'reviews_clean.csv')
payments    = pd.read_csv(CLEANED_PATH + 'payment_totals.csv')
products    = pd.read_csv(CLEANED_PATH + 'products_clean.csv')
sellers     = pd.read_csv(CLEANED_PATH + 'sellers_clean.csv')

delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Commandes livrées : {len(delivered):,} / {len(orders):,}')

## 1. KPIs de livraison

In [ ]:
kpis = {
    'Total commandes':            len(orders),
    'Commandes livrées':          len(delivered),
    'Taux livraison (%)':         round(len(delivered) / len(orders) * 100, 2),
    'Commandes en retard':        int(delivered['is_late'].sum()),
    'Taux de retard (%)':         round(delivered['is_late'].mean() * 100, 2),
    'Délai moyen réel (j)':       round(delivered['actual_delivery_days'].mean(), 1),
    'Délai médian réel (j)':      round(delivered['actual_delivery_days'].median(), 1),
    'Retard moyen si tard (j)':   round(delivered[delivered['is_late']==1]['delivery_delay_days'].mean(), 1),
    'Avance moyenne si à temps':  round(delivered[delivered['is_late']==0]['delivery_delay_days'].mean(), 1),
}

print('=== KPIs LIVRAISON ===')
for k, v in kpis.items():
    print(f'  {k:<35s}: {v:>10}')

## 2. KPIs satisfaction client

In [ ]:
merged = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

kpis_sat = {
    'Note moyenne globale':          round(merged['review_score'].mean(), 2),
    'Note moyenne — à temps':        round(merged[merged['is_late']==0]['review_score'].mean(), 2),
    'Note moyenne — en retard':      round(merged[merged['is_late']==1]['review_score'].mean(), 2),
    'Écart de note (pts)':           round(merged[merged['is_late']==0]['review_score'].mean() -
                                           merged[merged['is_late']==1]['review_score'].mean(), 2),
    'Taux 5 étoiles (%)':            round((reviews['review_score']==5).mean() * 100, 1),
    'Taux 1 étoile (%)':             round((reviews['review_score']==1).mean() * 100, 1),
}

print('=== KPIs SATISFACTION ===')
for k, v in kpis_sat.items():
    print(f'  {k:<40s}: {v:>8}')

## 3. Note par tranche de retard

In [ ]:
merged['delay_bucket'] = pd.cut(
    merged['delivery_delay_days'],
    bins=[-999, -14, -7, 0, 7, 14, 999],
    labels=['Avance > 14j', 'Avance 7-14j', 'Avance 0-7j', 'Retard 1-7j', 'Retard 8-14j', 'Retard > 14j']
)

delay_score = merged.groupby('delay_bucket', observed=False).agg(
    note_moyenne=('review_score', 'mean'),
    nb_commandes=('order_id', 'count')
).round(2)

print(delay_score)

fig, ax = plt.subplots(figsize=(10, 4))
colors = [GREEN, GREEN, GREEN, ORANGE, RED, RED]
bars = ax.bar(delay_score.index, delay_score['note_moyenne'], color=colors, edgecolor='none', width=0.6)
ax.axhline(y=4.09, color=GRAY, linestyle='--', linewidth=1, label='Moyenne globale (4,09)')
ax.set_ylim(0, 5.2)
ax.set_title('Note client moyenne par tranche de retard de livraison', fontsize=13, pad=12)
ax.set_ylabel('Note moyenne (/ 5)')
ax.set_xlabel('')
ax.legend(fontsize=10)
for bar, val in zip(bars, delay_score['note_moyenne']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_PATH + '03_note_par_retard.png', dpi=150)
plt.show()

## 4. Tendance mensuelle des retards

In [ ]:
delivered['month'] = delivered['order_purchase_timestamp'].dt.to_period('M')

monthly = delivered.groupby('month').agg(
    nb_commandes=('order_id', 'count'),
    taux_retard=('is_late', 'mean'),
    delai_moyen=('actual_delivery_days', 'mean')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly['taux_retard_pct'] = monthly['taux_retard'] * 100

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.bar(monthly['month_str'], monthly['nb_commandes'], color=BLUE, alpha=0.7, edgecolor='none', label='Commandes')
ax2.plot(monthly['month_str'], monthly['taux_retard_pct'], color=RED, marker='o', linewidth=2,
         markersize=4, label='Taux de retard (%)')

ax1.set_xlabel('')
ax1.set_ylabel('Nombre de commandes', color=BLUE)
ax2.set_ylabel('Taux de retard (%)', color=RED)
ax1.set_title('Volume mensuel de commandes et taux de retard', fontsize=13, pad=12)
ax1.tick_params(axis='x', rotation=45)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)

plt.tight_layout()
plt.savefig(DOCS_PATH + '03_tendance_mensuelle.png', dpi=150)
plt.show()

## 5. Taux de retard par état brésilien

In [ ]:
state_df = delivered.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

state_perf = state_df.groupby('customer_state').agg(
    nb_commandes=('order_id', 'count'),
    taux_retard=('is_late', 'mean'),
    delai_moyen=('actual_delivery_days', 'mean')
).round(3).sort_values('taux_retard', ascending=False)

state_perf['taux_retard_pct'] = (state_perf['taux_retard'] * 100).round(1)
print('Top 10 états avec le plus fort taux de retard :')
print(state_perf.head(10).to_string())

top10_states = state_perf.head(10)
colors = [RED if v > 15 else ORANGE if v > 12 else BLUE for v in top10_states['taux_retard_pct']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10_states.index, top10_states['taux_retard_pct'], color=colors, edgecolor='none')
ax.set_title('Top 10 états par taux de retard de livraison', fontsize=13, pad=12)
ax.set_xlabel('Taux de retard (%)')
for bar, val in zip(bars, top10_states['taux_retard_pct']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(DOCS_PATH + '03_retard_par_etat.png', dpi=150)
plt.show()

# Export pour Power BI
state_perf.reset_index().to_csv(CLEANED_PATH + 'kpi_by_state.csv', index=False)
print('kpi_by_state.csv exporté.')

## 6. Analyse par catégorie de produit

In [ ]:
items_full = items.merge(products[['product_id', 'product_category_name_english', 'product_weight_g']], on='product_id', how='left')
items_full = items_full.merge(delivered[['order_id', 'is_late', 'delivery_delay_days', 'actual_delivery_days']], on='order_id', how='inner')
items_full = items_full.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

cat_perf = items_full.groupby('product_category_name_english').agg(
    ca_total=('price', 'sum'),
    nb_commandes=('order_id', 'nunique'),
    taux_retard=('is_late', 'mean'),
    note_moyenne=('review_score', 'mean'),
    poids_moyen=('product_weight_g', 'mean')
).round(2)

cat_perf_filtered = cat_perf[cat_perf['nb_commandes'] >= 100].sort_values('ca_total', ascending=False)

print('Top 10 catégories par CA :')
print(cat_perf_filtered.head(10).to_string())

cat_perf_filtered.reset_index().to_csv(CLEANED_PATH + 'kpi_by_category.csv', index=False)
print('\nkpi_by_category.csv exporté.')

## 7. Performance des vendeurs

In [ ]:
seller_perf = items.merge(delivered[['order_id', 'is_late', 'delivery_delay_days', 'actual_delivery_days']], on='order_id', how='inner')
seller_perf = seller_perf.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

seller_stats = seller_perf.groupby('seller_id').agg(
    nb_commandes=('order_id', 'nunique'),
    ca_total=('price', 'sum'),
    taux_retard=('is_late', 'mean'),
    note_moyenne=('review_score', 'mean'),
    delai_moyen=('actual_delivery_days', 'mean')
).round(2)

# Classement par nombre de commandes
top_sellers = seller_stats.sort_values('nb_commandes', ascending=False).head(20)
print('Top 20 vendeurs par volume :')
print(top_sellers.to_string())

seller_stats.reset_index().to_csv(CLEANED_PATH + 'kpi_by_seller.csv', index=False)
print('\nkpi_by_seller.csv exporté.')

## 8. Corrélation poids produit / retard

In [ ]:
weight_df = items_full[['product_weight_g', 'is_late', 'actual_delivery_days']].dropna()

avg_weight = weight_df.groupby('is_late')['product_weight_g'].mean()
print('Poids moyen par statut livraison :')
print(f'  À temps   : {avg_weight[0]:,.0f} g')
print(f'  En retard : {avg_weight[1]:,.0f} g')

corr = weight_df['product_weight_g'].corr(weight_df['actual_delivery_days'])
print(f'\nCorrélation poids / délai réel : {corr:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['À temps', 'En retard'], [avg_weight[0], avg_weight[1]],
       color=[GREEN, RED], edgecolor='none', width=0.5)
ax.set_title('Poids moyen des produits selon le statut de livraison', fontsize=12, pad=10)
ax.set_ylabel('Poids moyen (g)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} g'))
for i, v in enumerate([avg_weight[0], avg_weight[1]]):
    ax.text(i, v + 30, f'{int(v):,} g', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(DOCS_PATH + '03_poids_vs_retard.png', dpi=150)
plt.show()

## 9. Export table analytique principale (pour Power BI & SQL)

In [ ]:
# Table principale : 1 ligne = 1 commande avec toutes les dimensions utiles
main_table = delivered.merge(
    customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left'
).merge(
    reviews[['order_id', 'review_score']], on='order_id', how='left'
).merge(
    payments[['order_id', 'total_payment', 'payment_type', 'nb_installments']], on='order_id', how='left'
)

# Agrégation des items par commande
items_agg = items.groupby('order_id').agg(
    nb_items=('order_item_id', 'count'),
    prix_total=('price', 'sum'),
    frais_port_total=('freight_value', 'sum')
).reset_index()

main_table = main_table.merge(items_agg, on='order_id', how='left')

cols_export = [
    'order_id', 'customer_id', 'customer_state', 'customer_city',
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'actual_delivery_days',
    'delivery_delay_days', 'is_late', 'carrier_delay_days',
    'purchase_year', 'purchase_month', 'purchase_quarter',
    'review_score', 'payment_type', 'total_payment', 'nb_installments',
    'nb_items', 'prix_total', 'frais_port_total'
]

main_table = main_table[cols_export]
main_table.to_csv(CLEANED_PATH + 'main_analysis_table.csv', index=False)

print(f'Table analytique principale exportée : {len(main_table):,} lignes x {len(main_table.columns)} colonnes')
print('Colonnes :', list(main_table.columns))

## 10. Résumé final

Tous les fichiers exportés dans `data/cleaned/` :
- `orders_clean.csv` — commandes avec variables métier
- `reviews_clean.csv` — avis dédupliqués
- `products_clean.csv` — produits avec catégories traduites
- `payments_clean.csv` — paiements nettoyés
- `payment_totals.csv` — agrégat paiements par commande
- `kpi_by_state.csv` — KPIs par état
- `kpi_by_category.csv` — KPIs par catégorie
- `kpi_by_seller.csv` — KPIs par vendeur
- `main_analysis_table.csv` — **table principale pour Power BI**

→ Prochaine étape : **`sql/01_create_tables.sql`** pour charger tout ça dans MySQL.